## Creating Spark Session

In [64]:
import os
import sys

In [65]:
# 1. Install Dependencies
!pip install -q pyspark findspark openml pyarrow

In [66]:
#This searches your system for an Apache Spark installation and configures your Python environment to recognize where the Spark libraries reside.
import findspark
findspark.init()

In [67]:
#SparkSession: This is the universal entry point for programming Spark with the Dataset and DataFrame AP
from pyspark.sql import SparkSession 
#These are specialized, highly optimized functions that execute on distributed Spark data nodes
from pyspark.sql.functions import *
#from pyspark.sql.window import Window
from pyspark.sql.window import Window
#Pipeline: A sequence of data preprocessing steps and algorithms chained together to automate and organize a machine learning workflow.
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
import pandas as pd
import openml

spark = SparkSession.builder \
    .appName("Enterprise_Pipeline") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .getOrCreate()


In [68]:
#Create local folder for keep data output
for folder in ['data/bronze', 'data/silver', 'data/gold']:
    os.makedirs(folder, exist_ok=True)

## Task 1: Data Ingestion & Exploration

## Reading Data 

In [69]:
df = spark.read.csv(
    "BNPParibas_Data.csv",
    header=True,
    inferSchema=True
)


In [70]:
##2. COMMIT UNTOUCHED RAW DATA TO BRONZE LAYER
# We use Pandas to write a clean CSV file. This completely avoids the winutils.exe requirement!
bronze_csv_path = R"data\bronze\raw_data.csv"
df.toPandas().to_csv(bronze_csv_path, index=False)

## Display :
Schema

Record Count

Null 

Duplicate Count

Data Types


In [71]:
df.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- age: integer (nullable = true)
 |-- tenure_months: integer (nullable = true)
 |-- monthly_charges: double (nullable = true)
 |-- total_charges: double (nullable = true)
 |-- contract_type: string (nullable = true)
 |-- internet_service: string (nullable = true)
 |-- support_tickets: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- churn: integer (nullable = true)



Record Count


In [72]:
df.count()

1000

In [73]:
df.describe().show()

+-------+-----------------+------------------+------------------+-----------------+------------------+--------------+----------------+------------------+--------------+------------------+
|summary|      customer_id|               age|     tenure_months|  monthly_charges|     total_charges| contract_type|internet_service|   support_tickets|payment_method|             churn|
+-------+-----------------+------------------+------------------+-----------------+------------------+--------------+----------------+------------------+--------------+------------------+
|  count|             1000|              1000|              1000|             1000|              1000|          1000|            1000|              1000|          1000|              1000|
|   mean|            500.5|            43.819|            35.459|79.96715000000002| 2800.235379999997|          NULL|            NULL|             1.956|          NULL|             0.502|
| stddev|288.8194360957494|14.991029650093076|20.36819044937

Null Count

In [74]:
df.select([
    count(when(col(c).isNull(),c)).alias(c)
    for c in df.columns]).show()


+-----------+---+-------------+---------------+-------------+-------------+----------------+---------------+--------------+-----+
|customer_id|age|tenure_months|monthly_charges|total_charges|contract_type|internet_service|support_tickets|payment_method|churn|
+-----------+---+-------------+---------------+-------------+-------------+----------------+---------------+--------------+-----+
|          0|  0|            0|              0|            0|            0|               0|              0|             0|    0|
+-----------+---+-------------+---------------+-------------+-------------+----------------+---------------+--------------+-----+



Duplicate Count

In [75]:
total_count=df.count()
unique_count=df.drop_duplicates().count()
duplicate_count=total_count-unique_count
duplicate_count

0

Data Types

In [76]:
df.dtypes

[('customer_id', 'int'),
 ('age', 'int'),
 ('tenure_months', 'int'),
 ('monthly_charges', 'double'),
 ('total_charges', 'double'),
 ('contract_type', 'string'),
 ('internet_service', 'string'),
 ('support_tickets', 'int'),
 ('payment_method', 'string'),
 ('churn', 'int')]

## Task 2: ETL Pipeline Development


In [77]:
#EXTRACT (Read from the Bronze Source Layer)
bronze_source_path = r"data\bronze\raw_data.csv"

In [78]:
df_raw = spark.read.csv(bronze_source_path, header=True, inferSchema=True)
initial_count = df_raw.count()
initial_count

1000

## Transform
Perform:

Missing Value Treatment

Duplicate Removal

Data Type Conversion

Feature Engineering

Aggregation


In [79]:
#Duplicate Removal
df_deduped = df_raw.dropDuplicates()
deduped_count = df_deduped.count()

In [80]:
df_transformed = df_deduped
df_transformed.show()

+-----------+---+-------------+---------------+-------------+--------------+----------------+---------------+--------------+-----+
|customer_id|age|tenure_months|monthly_charges|total_charges| contract_type|internet_service|support_tickets|payment_method|churn|
+-----------+---+-------------+---------------+-------------+--------------+----------------+---------------+--------------+-----+
|          1| 56|           15|          59.23|       929.62|Month-to-Month|           Fiber|              5|           UPI|    1|
|          9| 36|           17|          53.58|       962.59|      One Year|           Fiber|              1|          Cash|    0|
|         10| 40|           13|          95.45|      1229.25|      One Year|             DSL|              2|           UPI|    0|
|         40| 69|            9|          43.16|       397.59|      One Year|             DSL|              2|   Credit Card|    0|
|         44| 24|           36|          69.32|      2666.12|Month-to-Month|       

In [81]:
# Dynamically identify numeric vs categorical columns for automated processing
df_raw.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- age: integer (nullable = true)
 |-- tenure_months: integer (nullable = true)
 |-- monthly_charges: double (nullable = true)
 |-- total_charges: double (nullable = true)
 |-- contract_type: string (nullable = true)
 |-- internet_service: string (nullable = true)
 |-- support_tickets: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- churn: integer (nullable = true)



[data\bronze\raw_data.csv]  <-- Raw Unprocessed Ingestion Source

               │

               ▼

       ┌───────────────┐
       │   EXTRACT     │  ──> PySpark Ingestion Engine
       └───────────────┘

               │

               ▼

       ┌───────────────┐
       │   TRANSFORM   │  ──> 1. dropDuplicates()
       │               │  ──> 2. approxQuantile() Median Imputation
       │               │  ──> 3. UNKNOWN Categorical String Imputation
       │               │  ──> 4. Dynamic Column Ratio Engineering
       └───────────────┘
               │

               ▼
               
       ┌───────────────┐
       │     LOAD      │  ──> Native Pandas File Writer Workaround
       └───────────────┘
               │
               ▼
    [data\silver\cleaned_data.csv] <-- Cleaned, Production-Ready Dataset